In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [21]:
import math 

class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        query = self.W_q(x)
        value = self.W_v(x)

        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        mask = self.mask[:seq_len, :seq_len] == 0
        
        attention_matrix.masked_fill_(mask, float('-inf'))
        attention_matrix = torch.softmax(attention_matrix, 2) 
        
        return attention_matrix @ value
        


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.head_attentions = nn.ModuleList([HeadAttention(self.emb_size, self.head_size, self.max_seq_len) for i in range(self.num_heads)])
        self.otpt = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):

        tensors = [ head_attention(x) for head_attention in self.head_attentions]
        tensor = torch.cat(tensors, dim=2)
        tensor = self.otpt(tensor)
        tensor = self.dropout(tensor)
        return tensor
            

In [22]:
torch.manual_seed(42)
MultiHeadAttention(2,3,4,5).forward(torch.Tensor([[[1,2,3], [4,5,6]], [[6,5,4], [3,2,1]]]))

torch.Size([2, 2, 8])


tensor([[[-0.7020, -0.0000, -0.0000],
         [-0.5516, -2.2138, -0.7437]],

        [[-0.1699, -0.0000, -1.7349],
         [ 0.0704, -0.0000, -1.3123]]], grad_fn=<MulBackward0>)